# 03 Projeção Lexical sobre o DDlarge

DDlarge: 184k registros JSONL. Campo de texto: `relato` (mapeado para `text` pelo loader).

Implementação usa `spaCy PhraseMatcher` (Aho-Corasick) — patterns compilados uma vez, matching token-based (fronteiras de palavra grátis).

In [5]:
import json
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.data_loading import stream_ddlarge
from src.projection import compile_lexicon, annotate_record

ROOT = Path('..')
LEXICON_PATH = ROOT / 'data/lexicon/lexicon_filtered.json'
DDLARGE_PATH = ROOT / 'data/raw/DDlarge/dd_corpus_large.jsonl'
OUTPUT_PATH  = ROOT / 'data/processed/ddlarge_pseudo.jsonl'

with open(LEXICON_PATH, encoding='utf-8') as f:
    lexicon = json.load(f)

# Pre-compila PhraseMatcher UMA vez não dentro do loop de 184k registros
compiled = compile_lexicon(lexicon)
print(f'Léxico compilado: {len(lexicon)} entidades')

Léxico compilado: 1057 entidades


In [6]:
# Projeção 184k iterações
pseudo_records = []
label_counts = Counter()

for record in tqdm(stream_ddlarge(DDLARGE_PATH)):
    annotated = annotate_record(record, compiled)
    if annotated:
        pseudo_records.append(annotated)
        for s in annotated['spans']:
            label_counts[s['label']] += 1

print(f'Relatos pseudo-anotados: {len(pseudo_records)}')
print(f'Pseudo-entidades por classe: {dict(label_counts)}')

184493it [00:23, 7726.20it/s]

Relatos pseudo-anotados: 133841
Pseudo-entidades por classe: {'Location': 236763, 'Organization': 142636, 'Person': 71434}


In [7]:
# Exemplos de relatos pseudo-anotados
for rec in pseudo_records[:3]:
    print(rec['text'])
    for s in rec['spans']:
        print(f"  [{s['label']}] '{rec['text'][s['start']:s['end']]}'")
    print()

Dois moleques em uma moto estão tocando o terror em São Gonçalo, dois deram cobertura no assalto a um supermercado rede economia próximo a RJ nas próximidades do Guaxindiba e Jardim Catarina, a placa da motocicleta foi anotada: kzb2782 os dois entraram no Jardim Catarina
  [Location] 'São Gonçalo'
  [Location] 'RJ'
  [Location] 'Guaxindiba'
  [Location] 'Jardim Catarina'
  [Location] 'Jardim Catarina'

Traficantes do complexo Lins em posse do auto de placa KZN 6332,mesmo com upp na região os meliantes praticam assaltos na região da serra Grajaú-Jacarepaguá,Grajaú,vila Isabel e Jacarepaguá! Os mesmos portam fuzis,pistolas e Granadas!
  [Location] 'Lins'
  [Location] 'posse'
  [Organization] 'upp'
  [Location] 'Grajaú'
  [Location] 'vila Isabel'
  [Location] 'Jacarepaguá'

Melícia da estrada da pedra,matando inocentes e destruindo famílias.
Dodô,felipe branco,e Sandro são alguns que comandam e matam pessoas de bem daquela região.
Peço as autoridades que dem um jeito nessas pessoas.
  [Or

In [8]:
# Salvar corpus pseudo-anotado
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for rec in pseudo_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f'Salvo em {OUTPUT_PATH}')

Salvo em ..\data\processed\ddlarge_pseudo.jsonl
